In [1]:
!pip install sdv

In [2]:
import pandas as pd
import numpy as np
import os
import gc

from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata

In [3]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
BASE_PATH = "/content/drive/MyDrive/MajorProject"

TRAIN_PATH = f"{BASE_PATH}/CIC_IoT_2023/Full_Dataset/train_working.csv"
GAN_OUT_PATH = f"{BASE_PATH}/IDS_GAN_Outputs"

LABEL_COL = "label"
RANDOM_SEED = 42

In [5]:
print("Loading training data")
train_df = pd.read_csv(TRAIN_PATH, low_memory=False)

print("Initial shape", train_df.shape)

Loading training data
Initial shape (900000, 47)


In [6]:
train_df = train_df.dropna(subset=[LABEL_COL])
print("After label cleaning:", train_df.shape)

After label cleaning: (900000, 47)


In [7]:
GAN_TARGET_CLASSES = [
    "Recon-PingSweep",
    "Uploading_Attack",
    "SqlInjection",
    "BrowserHijacking",
    "Backdoor_Malware",
    "XSS"
]

gan_train_df = train_df[train_df[LABEL_COL].isin(GAN_TARGET_CLASSES)]

print("GAN training data shape:", gan_train_df.shape)
gan_train_df[LABEL_COL].value_counts()

GAN training data shape: (12851, 47)


,count
label,
BrowserHijacking,3522
SqlInjection,3104
XSS,2257
Backdoor_Malware,1892
Recon-PingSweep,1319
Uploading_Attack,757


In [8]:
# Create SDV metadata
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(gan_train_df)

metadata

{
    "columns": {
        "flow_duration": {
            "sdtype": "numerical"
        },
        "Header_Length": {
            "sdtype": "numerical"
        },
        "Protocol Type": {
            "sdtype": "numerical"
        },
        "Duration": {
            "sdtype": "numerical"
        },
        "Rate": {
            "sdtype": "numerical"
        },
        "Srate": {
            "sdtype": "numerical"
        },
        "Drate": {
            "sdtype": "categorical"
        },
        "fin_flag_number": {
            "sdtype": "categorical"
        },
        "syn_flag_number": {
            "sdtype": "categorical"
        },
        "rst_flag_number": {
            "sdtype": "categorical"
        },
        "psh_flag_number": {
            "sdtype": "categorical"
        },
        "ack_flag_number": {
            "sdtype": "categorical"
        },
        "ece_flag_number": {
            "sdtype": "categorical"
        },
        "cwr_flag_number": {
            "sdtype"

In [9]:
ctgan = CTGANSynthesizer(
    metadata=metadata,
    epochs=300,
    batch_size=2048,
    generator_lr=2e-4,
    discriminator_lr=2e-4,
    verbose=True,
    cuda=True,
    pac=1
)

/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:168: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/ctgan/synthesizers/_utils.py:16: FutureWarning: `cuda` parameter is deprecated and will be removed in a future release. Please use `enable_gpu` instead.
  warnings.warn(


In [10]:
print("Training CTGAN on minority classes")
ctgan.fit(gan_train_df)

Training CTGAN on minority classes


Gen. (-00.16) | Discrim. (-00.13): 100%|██████████| 300/300 [02:32<00:00,  1.97it/s]


In [11]:
real_counts = gan_train_df[LABEL_COL].value_counts()
real_counts

,count
label,
BrowserHijacking,3522
SqlInjection,3104
XSS,2257
Backdoor_Malware,1892
Recon-PingSweep,1319
Uploading_Attack,757


In [12]:
# generate upto 1x real samples per class
synthetic_counts = real_counts.to_dict()
synthetic_counts

{'BrowserHijacking': 3522,
 'SqlInjection': 3104,
 'XSS': 2257,
 'Backdoor_Malware': 1892,
 'Recon-PingSweep': 1319,
 'Uploading_Attack': 757}

## Generate synthetic data per class

In [13]:
synthetic_rows = []

for cls, n_samples in synthetic_counts.items():
    print(f"Generating {n_samples} samples for class {cls}")
    samples = ctgan.sample(num_rows=n_samples)
    samples[LABEL_COL] = cls

    synthetic_rows.append(samples)

synthetic_df = pd.concat(synthetic_rows, ignore_index=True)
print("Synthetic data shape:", synthetic_df.shape)

Generating 3522 samples for class BrowserHijacking
Generating 3104 samples for class SqlInjection
Generating 2257 samples for class XSS
Generating 1892 samples for class Backdoor_Malware
Generating 1319 samples for class Recon-PingSweep
Generating 757 samples for class Uploading_Attack
Synthetic data shape: (12851, 47)


## Save synthetic dataset

In [14]:
SYNTHETIC_PATH = f"{BASE_PATH}/CIC_IoT_2023/Full_Dataset/synthetic_minority_data.csv"
synthetic_df.to_csv(SYNTHETIC_PATH, index=False)

print("Synthetic data saved to:", SYNTHETIC_PATH)


Synthetic data saved to: /content/drive/MyDrive/MajorProject/CIC_IoT_2023/Full_Dataset/synthetic_minority_data.csv


In [15]:
augmented_train_df = pd.concat(
    [train_df, synthetic_df],
    ignore_index=True
)
print("Augmented data shape:", augmented_train_df.shape)

Augmented data shape: (912851, 47)


In [16]:
AUGMENTED_PATH = f"{BASE_PATH}/CIC_IoT_2023/Full_Dataset/augmented_train_data.csv"
augmented_train_df.to_csv(AUGMENTED_PATH, index=False)

print("Augmented data saved to: ", AUGMENTED_PATH)

Augmented data saved to:  /content/drive/MyDrive/MajorProject/CIC_IoT_2023/Full_Dataset/augmented_train_data.csv


In [17]:
del train_df, gan_train_df, synthetic_df, augmented_train_df
gc.collect()

0